# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
import os


base_url = "https://openai.vocareum.com/v1"
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_base=base_url,
    api_key=os.getenv('OPENAI_API_KEY')
)

In [8]:
# delete existing collection (if it exists)
try:
    chroma_client.delete_collection("udaplay")
except Exception as e:
    print("Could not delete collection (maybe it doesn't exist yet):", e)

# now safely create a new one
collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

## Query the database

In [10]:
print("Checking semantic search over the UdaPlay collection...\n")

example_queries = [
    "football games on PlayStation",
    "open world adventure",
    "classic Nintendo platformer",
    "fighting games",
    "racing games"
]

def demo_semantic_search(col, queries, top_k: int = 3):
    """Quick sanity check: run a few semantic queries and inspect results."""
    for q in queries:
        print(f"QUERY: {q}")
        print("-" * 50)
        try:
            result = col.query(
                query_texts=[q],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )

            # Chroma returns lists-of-lists because you can pass multiple queries
            docs = result.get("documents", [[]])[0]
            metas = result.get("metadatas", [[]])[0]
            dists = result.get("distances", [[]])[0]

            if not docs:
                print("  No matches found.\n")
                continue

            for rank, (meta, dist) in enumerate(zip(metas, dists), start=1):
                name = meta.get("Name", "Unknown")
                year = meta.get("YearOfRelease", "Unknown")
                platform = meta.get("Platform", "Unknown")
                genre = meta.get("Genre", "Unknown")

                # Distance is usually cosine distance; smaller is better
                print(f"  {rank}. {name} ({year})")
                print(f"     Platform: {platform} | Genre: {genre}")
                print(f"     Distance: {dist:.4f}")
            print()  # blank line between queries

        except Exception as e:
            print(f"  Error while querying collection: {e}\n")

print(f"Total documents in collection: {collection.count()}\n")
demo_semantic_search(collection, example_queries)

Checking semantic search over the UdaPlay collection...

Total documents in collection: 15

QUERY: football games on PlayStation
--------------------------------------------------
  1. Gran Turismo 5 (2010)
     Platform: PlayStation 3 | Genre: Racing
     Distance: 0.3692
  2. Gran Turismo (1997)
     Platform: PlayStation 1 | Genre: Racing
     Distance: 0.3929
  3. Marvel's Spider-Man (2018)
     Platform: PlayStation 4 | Genre: Action-adventure
     Distance: 0.4176

QUERY: open world adventure
--------------------------------------------------
  1. Marvel's Spider-Man (2018)
     Platform: PlayStation 4 | Genre: Action-adventure
     Distance: 0.3457
  2. Minecraft (2014)
     Platform: Xbox One | Genre: Sandbox, Survival
     Distance: 0.3473
  3. Grand Theft Auto: San Andreas (2004)
     Platform: PlayStation 2 | Genre: Action-adventure
     Distance: 0.3770

QUERY: classic Nintendo platformer
--------------------------------------------------
  1. Super Mario 64 (1996)
     Pla